# LSD1 CADD Project - Phases 5 and 6

**Prerequisite:** run `LSD1_CADD_Project.ipynb` first. This notebook loads
`LSD1_fragments.csv` produced by Phase 4.

| Phase | Content |
|---|---|
| 5 | Molecular similarity, library screening, hit selection |
| 6 | Molecular docking, enrichment, interaction analysis |

**Structure used:** PDB **6TUY** - human LSD1/CoREST with FAD and the
reversible quinazoline inhibitor MC4106 (ligand code NY8), 2.60 A.

In [ ]:
!pip install -q rdkit requests
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = "/content/drive/MyDrive/CADD_LSD1"
except Exception as e:
    print("Drive unavailable (", type(e).__name__, ") - using session storage.")
    PROJECT = "/content/CADD_LSD1"

os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
print("Working in:", os.getcwd())
print("\nFiles present:")
for f in sorted(os.listdir(".")):
    print("  ", f)

---
# Phase 5 - Molecular Similarity and Library Screening

## 5.1 Morgan fingerprints

Each molecule is encoded as a 2048-bit Morgan (ECFP4) fingerprint -
a binary vector recording which circular substructures are present.

Similarity between two molecules is the **Tanimoto coefficient**: the
number of bits set in both, divided by the number set in either. It runs
from 0 (nothing in common) to 1 (identical fingerprint).

In [ ]:
import pandas as pd, numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

ORDER = ["Active", "Moderate", "Inactive"]
df = pd.read_csv("LSD1_fragments.csv")
print("Loaded:", df.shape)
print(df["Class"].value_counts().reindex(ORDER).to_string())

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def fp_of(smi):
    m = Chem.MolFromSmiles(smi)
    return gen.GetFingerprint(m) if m is not None else None

df["fp"] = df["clean_smiles"].apply(fp_of)
df = df[df["fp"].notna()].reset_index(drop=True)
print("\nFingerprints generated:", len(df))

## 5.2 Are the active compounds structurally similar to each other?

This determines how Phase 5 should proceed. Two possibilities:

- **High mutual similarity** - the actives form one chemical series, and a
  similarity search will find close analogues of a single chemotype.
- **Low mutual similarity** - the actives span several distinct scaffolds,
  and any single query molecule will only retrieve part of the space.

Two distributions are compared: similarity among actives, and similarity
between actives and inactives. If the first is not clearly higher than the
second, structure alone does not separate potency.

In [ ]:
act_fps = list(df.loc[df["Class"] == "Active", "fp"])
ina_fps = list(df.loc[df["Class"] == "Inactive", "fp"])
print(f"Actives: {len(act_fps)}   Inactives: {len(ina_fps)}")

# Active vs Active (upper triangle only)
aa = []
for i in range(1, len(act_fps)):
    aa.extend(DataStructs.BulkTanimotoSimilarity(act_fps[i], act_fps[:i]))
aa = np.array(aa)

# Active vs Inactive (all cross pairs)
ai = []
for f in act_fps:
    ai.extend(DataStructs.BulkTanimotoSimilarity(f, ina_fps))
ai = np.array(ai)

print(f"\nActive-Active pairs   : {len(aa):,}   mean {aa.mean():.3f}   median {np.median(aa):.3f}")
print(f"Active-Inactive pairs : {len(ai):,}   mean {ai.mean():.3f}   median {np.median(ai):.3f}")

for t in (0.5, 0.7, 0.85, 0.9):
    print(f"  pairs >= {t}:  Act-Act {100*(aa>=t).mean():6.2f}%   "
          f"Act-Inact {100*(ai>=t).mean():6.2f}%")

In [ ]:
import matplotlib.pyplot as plt

GRID, AXIS, INK, MUTED = "#e1e0d9", "#c3c2b7", "#0b0b0b", "#898781"

fig, ax = plt.subplots(figsize=(8.5, 4.8), facecolor="#fcfcfb")
bins = np.linspace(0, 1, 60)
ax.hist(aa, bins=bins, density=True, alpha=0.60, color="#2a78d6",
        label=f"Active-Active (n={len(aa):,})")
ax.hist(ai, bins=bins, density=True, alpha=0.60, color="#1baf7a",
        label=f"Active-Inactive (n={len(ai):,})")
ax.set_facecolor("#fcfcfb")
ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
for s in ("left", "bottom"): ax.spines[s].set_color(AXIS)
ax.tick_params(colors=MUTED, labelsize=9)
ax.set_xlabel("Tanimoto similarity", color=INK)
ax.set_ylabel("Density", color=INK)
ax.set_title("Pairwise structural similarity", color=INK, fontsize=12, loc="left")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig("fig5_similarity_distributions.png", dpi=300, facecolor="#fcfcfb")
plt.show()

## 5.3 Scaffold diversity among the actives

The number of distinct Bemis-Murcko scaffolds, and how much of the active
set the most common scaffolds account for. A small number of scaffolds
covering most actives indicates a few dominant chemical series.

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold

def murcko(smi):
    try:
        s = MurckoScaffold.MurckoScaffoldSmiles(smi)
        return s if s else None
    except Exception:
        return None

act = df[df["Class"] == "Active"].copy()
act["scaffold"] = act["clean_smiles"].apply(murcko)
act = act.dropna(subset=["scaffold"])

vc = act["scaffold"].value_counts()
print(f"Active compounds      : {len(act)}")
print(f"Distinct scaffolds    : {len(vc)}")
print(f"Singleton scaffolds   : {(vc == 1).sum()}  ({100*(vc==1).sum()/len(vc):.0f}% of scaffolds)")
print(f"\nCoverage by most common scaffolds:")
cum = vc.cumsum() / len(act) * 100
for k in (1, 5, 10, 20, 50):
    if k <= len(vc):
        print(f"  top {k:>3} scaffolds cover {cum.iloc[k-1]:5.1f}% of actives")

## 5.4 Download the screening library (ZINC)

Enamine's Discovery Diversity Set requires registered access. An equivalent
set of commercially available compounds is obtained from **ZINC**, which
aggregates supplier catalogues - including Enamine's.

Tranches spanning drug-like molecular weight and logP ranges are downloaded
and pooled. A diversity set would be a poor choice here: it is constructed
to minimise near-neighbours, and would return almost nothing at the
similarity thresholds used below.

In [ ]:
import requests, io

TRANCHES = ["CBAA", "DCAA", "DCAB", "EBAB", "ECAB",
            "FCAB", "FDAB", "GCAB", "GDAB", "HDAB"]

LIB = "screening_library.csv"
if not os.path.exists(LIB):
    frames = []
    for t in TRANCHES:
        url = f"https://files.docking.org/2D/{t[:2]}/{t}.smi"
        try:
            r = requests.get(url, timeout=180)
            if r.status_code == 200 and not r.text.lstrip().startswith("<"):
                part = pd.read_csv(io.StringIO(r.text), sep=r"\s+")
                frames.append(part)
                print(f"  {t}: {len(part):,}")
            else:
                print(f"  {t}: unavailable")
        except Exception as e:
            print(f"  {t}: {type(e).__name__}")
    lib = pd.concat(frames, ignore_index=True).drop_duplicates("smiles")
    lib.to_csv(LIB, index=False)
else:
    lib = pd.read_csv(LIB)

print(f"\nLibrary compounds: {len(lib):,}")
lib.head()

## 5.5 Download the drug repurposing library

The **Broad Drug Repurposing Hub** catalogues approved and clinical-stage
drugs with structures and mechanism-of-action annotations. Screening it
alongside the ZINC library allows a direct comparison: do novel compounds
or existing drugs rank better against LSD1?

In [ ]:
BASE = "https://s3.amazonaws.com/data.clue.io/repurposing/downloads"

def load_broad(fname):
    txt = requests.get(f"{BASE}/{fname}", timeout=180).text
    lines = [l for l in txt.splitlines() if not l.startswith("!")]
    return pd.read_csv(io.StringIO("\n".join(lines)), sep="\t")

samples = load_broad("repurposing_samples_20200324.txt")
drugs   = load_broad("repurposing_drugs_20200324.txt")

print("samples:", samples.shape, "| columns:", list(samples.columns)[:8])
print("drugs  :", drugs.shape,   "| columns:", list(drugs.columns)[:8])

dl = (samples[["pert_iname", "smiles"]]
      .dropna()
      .drop_duplicates("pert_iname")
      .reset_index(drop=True))

ann = drugs[["pert_iname", "clinical_phase", "moa", "target"]].drop_duplicates("pert_iname")
dl = dl.merge(ann, on="pert_iname", how="left")

print(f"\nUnique drugs with SMILES: {len(dl):,}")
print(dl["clinical_phase"].value_counts().head(8).to_string())
dl.to_csv("drug_library.csv", index=False)
dl.head()

## 5.6 Similarity screening with a threshold sweep

Every library compound is compared against every active compound; its score
is the **maximum** Tanimoto similarity to any active.

The project brief specifies a 0.90 cutoff. Rather than reporting a single
number, the hit count is tabulated across a range of thresholds. This
characterises the library properly and identifies the highest cutoff that
still yields a workable number of compounds.

In [ ]:
def screen(smiles_list, query_fps, label, chunk=5000):
    """Max Tanimoto similarity of each library compound to any query."""
    best = np.zeros(len(smiles_list), dtype=float)
    valid = np.zeros(len(smiles_list), dtype=bool)
    for start in range(0, len(smiles_list), chunk):
        block = smiles_list[start:start + chunk]
        for j, smi in enumerate(block):
            m = Chem.MolFromSmiles(smi)
            if m is None:
                continue
            f = gen.GetFingerprint(m)
            best[start + j] = max(DataStructs.BulkTanimotoSimilarity(f, query_fps))
            valid[start + j] = True
        print(f"  {label}: {min(start+chunk, len(smiles_list)):,}/{len(smiles_list):,}")
    return best, valid

zinc_sim, zinc_ok = screen(lib["smiles"].tolist(), act_fps, "ZINC")
lib["max_sim"] = zinc_sim
lib = lib[zinc_ok].copy()

drug_sim, drug_ok = screen(dl["smiles"].tolist(), act_fps, "drugs")
dl["max_sim"] = drug_sim
dl = dl[drug_ok].copy()

In [ ]:
print("THRESHOLD SWEEP\n")
print(f"{'cutoff':>8}  {'ZINC hits':>12}  {'Drug hits':>12}")
print("-" * 36)
for t in (0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50):
    print(f"{t:>8.2f}  {int((lib['max_sim']>=t).sum()):>12,}  "
          f"{int((dl['max_sim']>=t).sum()):>12,}")

print(f"\nZINC  max similarity observed: {lib['max_sim'].max():.3f}")
print(f"Drugs max similarity observed: {dl['max_sim'].max():.3f}")

print("\n\nTOP 15 DRUG HITS")
cols = ["pert_iname", "max_sim", "clinical_phase", "moa"]
print(dl.nlargest(15, "max_sim")[cols].to_string(index=False))

## 5.7 Select the working cutoff and cluster the hits

A catalogue library contains families of close analogues, so a raw hit list
is highly redundant. Hits are clustered with the **Butina** algorithm and one
representative - the most similar to an active - is taken from each cluster.

This yields a structurally diverse shortlist for docking rather than
many variations on one molecule.

In [ ]:
from rdkit.ML.Cluster import Butina

# Pick the highest cutoff that still returns a usable number of compounds
CUTOFF = None
for t in (0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60):
    if (lib["max_sim"] >= t).sum() >= 50:
        CUTOFF = t
        break
if CUTOFF is None:
    CUTOFF = 0.60
print("Working similarity cutoff:", CUTOFF)

hits = lib[lib["max_sim"] >= CUTOFF].copy().reset_index(drop=True)
print("Raw hits:", len(hits))

hit_fps = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in hits["smiles"]]

dists = []
for i in range(1, len(hit_fps)):
    sims = DataStructs.BulkTanimotoSimilarity(hit_fps[i], hit_fps[:i])
    dists.extend(1.0 - np.array(sims))

clusters = Butina.ClusterData(dists, len(hit_fps), 0.35, isDistData=True)
print("Clusters:", len(clusters))

reps = []
for cl in clusters:
    sub = hits.iloc[list(cl)]
    reps.append(sub["max_sim"].idxmax())
zinc_hits = hits.loc[reps].sort_values("max_sim", ascending=False).reset_index(drop=True)

print("Cluster representatives:", len(zinc_hits))
zinc_hits.to_csv("hits_zinc.csv", index=False)
zinc_hits.head(10)

In [ ]:
DRUG_CUTOFF = 0.40
drug_hits = (dl[dl["max_sim"] >= DRUG_CUTOFF]
             .sort_values("max_sim", ascending=False)
             .head(100)
             .reset_index(drop=True))

print(f"Drug hits at >= {DRUG_CUTOFF}: {len(drug_hits)}")
drug_hits.to_csv("hits_drugs.csv", index=False)
print(drug_hits[["pert_iname", "max_sim", "clinical_phase", "moa"]].head(20).to_string(index=False))

## 5.8 Assemble the docking set

Four groups are docked together:

| Group | Purpose |
|---|---|
| **Known actives** | Should rank highly if docking works |
| **Known inactives** | Should rank poorly - the negative control |
| **ZINC hits** | Novel candidate compounds |
| **Drug hits** | Repurposing candidates |

Including known actives and inactives is essential: without them the
docking scores cannot be validated, and the enrichment calculation in
Phase 6 would be impossible.

In [ ]:
N_ACT, N_INA = 100, 100

sel_act = (df[df["Class"] == "Active"]
           .nlargest(N_ACT, "pIC50")[["clean_smiles", "pIC50"]]
           .assign(group="active"))
sel_ina = (df[df["Class"] == "Inactive"]
           .sample(min(N_INA, (df["Class"] == "Inactive").sum()), random_state=42)
           [["clean_smiles", "pIC50"]]
           .assign(group="inactive"))

sel_zinc = (zinc_hits.head(100)
            .rename(columns={"smiles": "clean_smiles"})[["clean_smiles"]]
            .assign(pIC50=np.nan, group="zinc_hit"))
sel_drug = (drug_hits.head(100)
            .rename(columns={"smiles": "clean_smiles"})[["clean_smiles"]]
            .assign(pIC50=np.nan, group="drug_hit"))

dock_set = pd.concat([sel_act, sel_ina, sel_zinc, sel_drug], ignore_index=True)
dock_set = dock_set.drop_duplicates("clean_smiles").reset_index(drop=True)
dock_set["lig_id"] = [f"L{i:04d}" for i in range(len(dock_set))]

print(dock_set["group"].value_counts().to_string())
print("\nTotal ligands to dock:", len(dock_set))
dock_set.to_csv("docking_set.csv", index=False)

---
# Phase 6 - Molecular Docking

## 6.1 Install docking software

- **AutoDock Vina** - the docking engine
- **Meeko** - ligand preparation (torsions, atom typing, PDBQT output)
- **PDBFixer / OpenMM** - repair of missing protein atoms
- **Open Babel** - receptor PDBQT conversion
- **PLIP** - protein-ligand interaction profiling
- **py3Dmol** - in-notebook structure visualisation

Installation takes several minutes.

In [ ]:
!pip install -q vina meeko pdbfixer openmm plip py3Dmol biopython
!apt-get -qq install -y openbabel > /dev/null 2>&1
print("install step finished")

import subprocess
print(subprocess.run(["obabel", "-V"], capture_output=True, text=True).stdout.strip())
from vina import Vina
print("Vina python bindings OK")

## 6.2 Download and inspect the structure

**6TUY** - human LSD1/CoREST with FAD and the reversible quinazoline
inhibitor MC4106 (ligand code **NY8**), 2.60 A.

The contents are listed before any processing, so that what is kept and what
is discarded is explicit.

In [ ]:
PDB_ID   = "6TUY"
LIG_CODE = "NY8"

if not os.path.exists(f"{PDB_ID}.pdb"):
    r = requests.get(f"https://files.rcsb.org/download/{PDB_ID}.pdb", timeout=180)
    r.raise_for_status()
    open(f"{PDB_ID}.pdb", "w").write(r.text)

lines = open(f"{PDB_ID}.pdb").read().splitlines()
print([l for l in lines if l.startswith("TITLE")][0])

from collections import Counter
het = Counter(l[17:20].strip() for l in lines if l.startswith("HETATM"))
chains = Counter(l[21] for l in lines if l.startswith("ATOM"))

print("\nProtein chains (atom counts):", dict(chains))
print("\nHeterogens present:")
for k, v in het.most_common():
    tag = ""
    if k == "FAD":      tag = "  <-- COFACTOR, MUST KEEP"
    elif k == LIG_CODE: tag = "  <-- crystal ligand (reference pose)"
    elif k == "HOH":    tag = "  (water, removed)"
    print(f"   {k:<5} {v:>6} atoms{tag}")

## 6.3 Prepare the receptor - keeping FAD

**This is the step where LSD1 docking usually goes wrong.**

LSD1 is a flavin-dependent enzyme and FAD sits at the base of the inhibitor
binding site. Standard cleanup routines classify FAD as a heterogen and
delete it along with waters and buffer components. If that happens the
pocket becomes an open cavity and every docking score is meaningless -
with no error raised.

The protein and FAD are therefore handled separately: PDBFixer repairs the
protein only, and FAD is reattached afterwards. The final file is checked
for FAD before docking proceeds.

In [ ]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

# 1. split protein / FAD / crystal ligand
prot_lines = [l for l in lines if l.startswith(("ATOM", "TER"))]
fad_lines  = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == "FAD"]
lig_lines  = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == LIG_CODE]

print(f"protein atoms {len(prot_lines)} | FAD atoms {len(fad_lines)} | ligand atoms {len(lig_lines)}")
assert fad_lines, "FAD not found - stop and check the structure"
assert lig_lines, "crystal ligand not found"

open("protein_only.pdb", "w").write("\n".join(prot_lines) + "\nEND\n")
open("fad.pdb", "w").write("\n".join(fad_lines) + "\nEND\n")
open("crystal_ligand.pdb", "w").write("\n".join(lig_lines) + "\nEND\n")

# 2. repair the protein
fixer = PDBFixer(filename="protein_only.pdb")
fixer.findMissingResidues()
fixer.findMissingAtoms()
print("\nmissing residues:", len(fixer.missingResidues),
      "| missing atoms:", len(fixer.missingAtoms))
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.4)
PDBFile.writeFile(fixer.topology, fixer.positions,
                  open("protein_fixed.pdb", "w"), keepIds=True)

# 3. reattach FAD
fixed = [l for l in open("protein_fixed.pdb").read().splitlines()
         if l.startswith(("ATOM", "TER"))]
open("receptor.pdb", "w").write("\n".join(fixed + fad_lines) + "\nEND\n")

check = open("receptor.pdb").read()
print("\nFAD present in receptor.pdb:", "FAD" in check)
assert "FAD" in check, "FAD was lost during preparation"

In [ ]:
# Convert to PDBQT (rigid receptor)
!obabel receptor.pdb -O receptor.pdbqt -xr -p 7.4 2> /dev/null
n = sum(1 for l in open("receptor.pdbqt") if l.startswith(("ATOM", "HETATM")))
print("receptor.pdbqt atoms:", n)
print("FAD retained in PDBQT:", "FAD" in open("receptor.pdbqt").read())

## 6.4 Define the docking box

The search box is centred on the centroid of the crystal ligand, which marks
the experimentally observed binding site. A 22 A cube comfortably encloses
the pocket without inflating the search space.

In [ ]:
coords = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                   for l in lig_lines])
center = coords.mean(axis=0)
extent = coords.max(axis=0) - coords.min(axis=0)

BOX = [22.0, 22.0, 22.0]
CENTER = [round(float(c), 3) for c in center]

print("crystal ligand centroid :", CENTER)
print("crystal ligand extent   :", [round(float(e), 1) for e in extent])
print("docking box size        :", BOX)

## 6.5 Prepare ligands

Meeko converts each SMILES into a 3D PDBQT file with rotatable bonds
assigned. Molecules that fail embedding are reported and skipped.

In [ ]:
from rdkit.Chem import AllChem
from meeko import MoleculePreparation, PDBQTWriterLegacy

os.makedirs("ligands", exist_ok=True)
dock_set = pd.read_csv("docking_set.csv")

def make_pdbqt(smi, path):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return False
    m = Chem.AddHs(m)
    if AllChem.EmbedMolecule(m, randomSeed=0xC0FFEE) != 0:
        return False
    try:
        AllChem.MMFFOptimizeMolecule(m, maxIters=400)
    except Exception:
        pass
    prep = MoleculePreparation()
    setups = prep.prepare(m)
    if not setups:
        return False
    txt, ok, _ = PDBQTWriterLegacy.write_string(setups[0])
    if not ok:
        return False
    open(path, "w").write(txt)
    return True

ok_rows, failed = [], 0
for _, r in dock_set.iterrows():
    p = f"ligands/{r['lig_id']}.pdbqt"
    if os.path.exists(p) or make_pdbqt(r["clean_smiles"], p):
        ok_rows.append(r)
    else:
        failed += 1

prepared = pd.DataFrame(ok_rows).reset_index(drop=True)
print(f"prepared {len(prepared)} | failed {failed}")
print(prepared["group"].value_counts().to_string())
prepared.to_csv("docking_set_prepared.csv", index=False)

## 6.6 Protocol validation by redocking

Before docking unknown compounds, the protocol is tested on a molecule whose
answer is already known: the crystal ligand is extracted, stripped of its
coordinates, and docked back into the receptor.

The predicted pose is compared with the crystallographic pose by RMSD.

| RMSD | Interpretation |
|---|---|
| < 2.0 A | Protocol reproduces the experimental pose - acceptable |
| 2.0 - 3.0 A | Marginal; interpret results cautiously |
| > 3.0 A | Protocol is not reliable for this site |

**Any docking study without this check is uninterpretable.** It is the only
direct evidence that the setup - box, receptor preparation, cofactor
placement - is correct.

In [ ]:
from vina import Vina

# Build a clean 3D ligand from the crystal ligand's own connectivity
ref = Chem.MolFromPDBFile("crystal_ligand.pdb", removeHs=False, sanitize=False)
print("crystal ligand atoms:", ref.GetNumAtoms() if ref else None)

redock_ok = make_pdbqt(Chem.MolToSmiles(Chem.RemoveHs(ref)) if ref else "",
                       "ligands/REDOCK.pdbqt") if ref else False
print("redock ligand prepared:", redock_ok)

v = Vina(sf_name="vina", cpu=0, seed=42)
v.set_receptor("receptor.pdbqt")
v.compute_vina_maps(center=CENTER, box_size=BOX)
print("affinity maps computed")

if redock_ok:
    v.set_ligand_from_file("ligands/REDOCK.pdbqt")
    v.dock(exhaustiveness=16, n_poses=10)
    v.write_poses("redock_out.pdbqt", n_poses=1, overwrite=True)
    print("redock score:", round(v.energies(n_poses=1)[0][0], 2), "kcal/mol")

In [ ]:
# RMSD between the redocked pose and the crystallographic pose
!obabel redock_out.pdbqt -O redock_pose.pdb 2> /dev/null

pose = Chem.MolFromPDBFile("redock_pose.pdb", sanitize=False, removeHs=True)
xtal = Chem.MolFromPDBFile("crystal_ligand.pdb", sanitize=False, removeHs=True)

if pose and xtal and pose.GetNumAtoms() == xtal.GetNumAtoms():
    a = pose.GetConformer().GetPositions()
    b = xtal.GetConformer().GetPositions()
    rmsd = float(np.sqrt(((a - b) ** 2).sum(axis=1).mean()))
    print(f"Redocking RMSD: {rmsd:.2f} A")
    print("VERDICT:", "PASS" if rmsd < 2.0 else
                      "MARGINAL" if rmsd < 3.0 else "FAIL")
else:
    print("Atom counts differ - compare poses visually in 6.9 instead.")
    print("pose:", pose.GetNumAtoms() if pose else None,
          " xtal:", xtal.GetNumAtoms() if xtal else None)

## 6.7 Dock the full set

Each ligand is docked using the affinity maps already computed. Results are
appended to disk after every molecule, so an interrupted session can be
resumed without losing completed work.

Expect roughly 10-20 seconds per ligand. Several hundred ligands therefore
takes one to two hours - leave the cell running.

In [ ]:
import time, csv

RESULTS = "docking_results.csv"
done = set()
if os.path.exists(RESULTS):
    done = set(pd.read_csv(RESULTS)["lig_id"])
    print("resuming - already docked:", len(done))

prepared = pd.read_csv("docking_set_prepared.csv")
todo = prepared[~prepared["lig_id"].isin(done)]
print("to dock:", len(todo))

write_header = not os.path.exists(RESULTS)
t0 = time.time()
with open(RESULTS, "a", newline="") as fh:
    w = csv.writer(fh)
    if write_header:
        w.writerow(["lig_id", "group", "pIC50", "smiles", "score"])
    for i, (_, r) in enumerate(todo.iterrows(), 1):
        try:
            v.set_ligand_from_file(f"ligands/{r['lig_id']}.pdbqt")
            v.dock(exhaustiveness=8, n_poses=5)
            score = float(v.energies(n_poses=1)[0][0])
            v.write_poses(f"ligands/{r['lig_id']}_out.pdbqt",
                          n_poses=1, overwrite=True)
        except Exception as e:
            print("  fail", r["lig_id"], type(e).__name__)
            score = np.nan
        w.writerow([r["lig_id"], r["group"], r["pIC50"],
                    r["clean_smiles"], score])
        fh.flush()
        if i % 20 == 0:
            el = time.time() - t0
            print(f"  {i}/{len(todo)}   {el/i:.1f}s/ligand   "
                  f"ETA {(len(todo)-i)*el/i/60:.0f} min")

print("docking complete")

## 6.8 Enrichment analysis

The central question: **does docking rank known actives above known
inactives?**

If it does, the scores carry real signal and the novel hits are worth
taking seriously. If it does not, the docking protocol has no discriminative
power for this target and the ranking of new compounds means nothing.

Reported honestly either way - a negative result that is correctly
diagnosed is a legitimate finding.

In [ ]:
res = pd.read_csv("docking_results.csv").dropna(subset=["score"])
res = res.sort_values("score")          # more negative = better
res["rank"] = range(1, len(res) + 1)

print("Docked:", len(res))
print("\nScore by group (kcal/mol):")
print(res.groupby("group")["score"].describe()[["count","mean","50%","min"]].round(2).to_string())

n_act = (res["group"] == "active").sum()
n_ina = (res["group"] == "inactive").sum()

print("\n\nENRICHMENT")
print(f"{'top %':>7} {'n':>5} {'actives':>9} {'inactives':>11} {'EF':>7}")
print("-" * 44)
for pct in (1, 5, 10, 20):
    k = max(1, int(len(res) * pct / 100))
    top = res.head(k)
    a = (top["group"] == "active").sum()
    i = (top["group"] == "inactive").sum()
    expected = n_act * k / len(res)
    ef = a / expected if expected > 0 else np.nan
    print(f"{pct:>6}% {k:>5} {a:>9} {i:>11} {ef:>7.2f}")

k5 = max(1, int(len(res) * 0.05))
pct_in_top5 = 100 * (res.head(k5)["group"] == "active").sum() / n_act
print(f"\n{pct_in_top5:.1f}% of known actives fall in the top 5% of the ranking")

from scipy.stats import mannwhitneyu
sa = res.loc[res["group"] == "active", "score"]
si = res.loc[res["group"] == "inactive", "score"]
u, p = mannwhitneyu(sa, si, alternative="less")
print(f"Actives score better than inactives: p = {p:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8), facecolor="#fcfcfb")
GROUPS = ["active", "inactive", "zinc_hit", "drug_hit"]
COLS   = ["#2a78d6", "#1baf7a", "#eb6834", "#eda100"]
data = [res.loc[res["group"] == g, "score"] for g in GROUPS]
data = [d for d in data if len(d)]
labs = [g for g, d in zip(GROUPS, [res.loc[res["group"]==g,"score"] for g in GROUPS]) if len(d)]

bp = ax.boxplot(data, patch_artist=True, notch=True, widths=0.55,
                medianprops=dict(color=INK, linewidth=1.6))
for patch, c in zip(bp["boxes"], COLS):
    patch.set_facecolor(c); patch.set_alpha(0.65); patch.set_edgecolor(AXIS)
ax.set_xticks(range(1, len(labs) + 1)); ax.set_xticklabels(labs)
ax.set_facecolor("#fcfcfb"); ax.grid(True, color=GRID, linewidth=0.6)
ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
for s in ("left", "bottom"): ax.spines[s].set_color(AXIS)
ax.tick_params(colors=MUTED, labelsize=9)
ax.set_ylabel("Vina score (kcal/mol)", color=INK)
ax.set_title("Docking scores by compound group", color=INK, fontsize=12, loc="left")
plt.tight_layout()
plt.savefig("fig6_docking_scores.png", dpi=300, facecolor="#fcfcfb")
plt.show()

print("\nTOP 20 OVERALL")
print(res.head(20)[["rank","lig_id","group","score","pIC50"]].to_string(index=False))

## 6.9 Interaction analysis of the top compounds (PLIP)

For the five best-scoring compounds, the docked pose is combined with the
receptor and analysed with **PLIP**, which detects hydrogen bonds,
hydrophobic contacts, pi-stacking, salt bridges and halogen bonds.

Interactions involving **FAD** are of particular interest: contact with the
cofactor indicates binding in the catalytic site rather than at a surface
pocket.

In [ ]:
import subprocess
from plip.structure.preparation import PDBComplex

top5 = res.head(5)
report = []

for _, r in top5.iterrows():
    lid = r["lig_id"]
    subprocess.run(["obabel", f"ligands/{lid}_out.pdbqt",
                    "-O", f"ligands/{lid}_pose.pdb"],
                   capture_output=True)

    pose = [l for l in open(f"ligands/{lid}_pose.pdb").read().splitlines()
            if l.startswith(("ATOM", "HETATM"))]
    pose = [("HETATM" + l[6:17] + "LIG" + " " + "Z" + l[22:]) for l in pose]

    rec = [l for l in open("receptor.pdb").read().splitlines()
           if l.startswith(("ATOM", "HETATM"))]
    cpx = f"complex_{lid}.pdb"
    open(cpx, "w").write("\n".join(rec + pose) + "\nEND\n")

    try:
        m = PDBComplex()
        m.load_pdb(cpx)
        m.analyze()
        for key, site in m.interaction_sets.items():
            if "LIG" not in key:
                continue
            for hb in site.hbonds_pdon + site.hbonds_ldon:
                report.append(dict(lig=lid, score=r["score"], type="H-bond",
                                   residue=f"{hb.restype}{hb.resnr}",
                                   dist=round(hb.distance_ad, 2)))
            for hc in site.hydrophobic_contacts:
                report.append(dict(lig=lid, score=r["score"], type="Hydrophobic",
                                   residue=f"{hc.restype}{hc.resnr}",
                                   dist=round(hc.distance, 2)))
            for ps in site.pistacking:
                report.append(dict(lig=lid, score=r["score"], type="pi-stacking",
                                   residue=f"{ps.restype}{ps.resnr}",
                                   dist=round(ps.distance, 2)))
            for sb in site.saltbridge_lneg + site.saltbridge_pneg:
                report.append(dict(lig=lid, score=r["score"], type="Salt bridge",
                                   residue=f"{sb.restype}{sb.resnr}",
                                   dist=round(sb.distance, 2)))
    except Exception as e:
        print("PLIP failed for", lid, type(e).__name__, e)

inter = pd.DataFrame(report)
if len(inter):
    inter.to_csv("table4_interactions.csv", index=False)
    print(inter.to_string(index=False))
    print("\n\nInteraction counts per compound:")
    print(pd.crosstab(inter["lig"], inter["type"]).to_string())
    print("\nContacts involving FAD:")
    print(inter[inter["residue"].str.contains("FAD", na=False)].to_string(index=False))
else:
    print("No interactions parsed - inspect complex_*.pdb manually.")

## 6.10 Visualise the best pose

In [ ]:
import py3Dmol

best = res.iloc[0]["lig_id"]
view = py3Dmol.view(width=900, height=600)
view.addModel(open("receptor.pdb").read(), "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.addStyle({"resn": "FAD"},
              {"stick": {"colorscheme": "yellowCarbon", "radius": 0.18}})
view.addModel(open(f"ligands/{best}_pose.pdb").read(), "pdb")
view.setStyle({"model": -1},
              {"stick": {"colorscheme": "cyanCarbon", "radius": 0.22}})
view.zoomTo({"model": -1})
print(f"Best compound: {best}   score {res.iloc[0]['score']:.2f} kcal/mol")
print("yellow = FAD cofactor,  cyan = docked ligand")
view.show()

## 6.11 Final summary

In [ ]:
print("=" * 60)
print("LSD1 VIRTUAL SCREENING - SUMMARY")
print("=" * 60)
print(f"Target                : LSD1 / KDM1A (CHEMBL6136)")
print(f"Structure             : {PDB_ID}, FAD retained")
print(f"Curated compounds     : {len(df):,}")
print(f"Screening library     : {len(lib):,} (ZINC)")
print(f"Drug library          : {len(dl):,} (Broad Repurposing Hub)")
print(f"Similarity cutoff     : {CUTOFF}")
print(f"ZINC hits (clustered) : {len(zinc_hits)}")
print(f"Drug hits             : {len(drug_hits)}")
print(f"Ligands docked        : {len(res)}")
print(f"Actives in top 5%     : {pct_in_top5:.1f}%")

print("\nOutput files:")
for f in sorted(os.listdir(".")):
    if f.endswith((".csv", ".png")):
        print("   ", f)

---
## Interpretation checklist

1. **Did redocking pass?** If RMSD > 3 A, say so plainly and treat all
   rankings as unreliable.
2. **Is the enrichment factor above 1?** If actives are not ranked above
   inactives, docking has no discriminative power here - report that, and
   consider why (rigid receptor, no explicit waters, scoring-function limits).
3. **Do the top compounds contact FAD?** Contacts with the cofactor support
   binding in the catalytic site.
4. **Did any approved drug rank well?** Check its annotated mechanism in the
   Broad data - an unrelated mechanism scoring highly is more likely a
   docking artefact than a discovery.